# Logistic regression on balanced dataset

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib  # For saving the model

In [5]:
# Load CSV file
df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/balanced-dataset/balanced_dataset.csv")

# Quick check
print(df.head())
print(df.info())

   label                                               text
0      1  "@Blackman38Tide: @WhaleLookyHere @HowdyDowdy1...
1      1  "@CB_Baby24: @white_thunduh alsarabsss" hes a ...
2      1  "@DevilGrimz: @VigxRArts you're fucking gay, b...
3      1  "@MarkRoundtreeJr: LMFAOOOO I HATE BLACK PEOPL...
4      1  "@NoChillPaz: "At least I'm not a nigger" http...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466578 entries, 0 to 466577
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   466578 non-null  int64 
 1   text    466578 non-null  object
dtypes: int64(1), object(1)
memory usage: 7.1+ MB
None


In [7]:

# 2. Define features and target
# Suppose your text column is named 'text' and target column is 'label'
X = df['text']
y = df['label']

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words='english'
)
vectorizer.fit(X_train)

TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words='english')

In [8]:
from sklearn.linear_model import SGDClassifier
# -----------------------------
# 4. Initialize SGDClassifier (Logistic Regression)
# -----------------------------
clf = SGDClassifier(
    loss='log',         # Logistic Regression
    penalty='l2',
    alpha=1e-4,         # Regularization strength
    max_iter=1,         # We'll manually iterate over epochs
    warm_start=True,    # Keep model between calls to fit()
    n_jobs=-1,
    random_state=42
)

In [9]:
import time
clf = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1, warm_start=True, n_jobs=-1)
# -----------------------------
# 5. Training with mini-batches and progress info
# -----------------------------
batch_size = 5000  # Adjust depending on memory
epochs = 5         # Number of full passes over data
num_samples = X_train.shape[0]

print("Starting training...")

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    start_time_epoch = time.time()
    
    for start in range(0, num_samples, batch_size):
        end = min(start + batch_size, num_samples)
        
        X_batch = vectorizer.transform(X_train.iloc[start:end])
        y_batch = y_train.iloc[start:end]
        
        clf.partial_fit(X_batch, y_batch, classes=[0,1])
        
        print(f" Trained on samples {start+1} to {end} / {num_samples}")
    
    epoch_time = time.time() - start_time_epoch
    print(f" Epoch {epoch+1} completed in {epoch_time:.2f} seconds")



Starting training...

Epoch 1/5
 Trained on samples 1 to 5000 / 373262
 Trained on samples 5001 to 10000 / 373262
 Trained on samples 10001 to 15000 / 373262
 Trained on samples 15001 to 20000 / 373262
 Trained on samples 20001 to 25000 / 373262
 Trained on samples 25001 to 30000 / 373262
 Trained on samples 30001 to 35000 / 373262
 Trained on samples 35001 to 40000 / 373262
 Trained on samples 40001 to 45000 / 373262
 Trained on samples 45001 to 50000 / 373262
 Trained on samples 50001 to 55000 / 373262
 Trained on samples 55001 to 60000 / 373262
 Trained on samples 60001 to 65000 / 373262
 Trained on samples 65001 to 70000 / 373262
 Trained on samples 70001 to 75000 / 373262
 Trained on samples 75001 to 80000 / 373262
 Trained on samples 80001 to 85000 / 373262
 Trained on samples 85001 to 90000 / 373262
 Trained on samples 90001 to 95000 / 373262
 Trained on samples 95001 to 100000 / 373262
 Trained on samples 100001 to 105000 / 373262
 Trained on samples 105001 to 110000 / 373262
 

In [10]:
# -----------------------------
# 6. Evaluate on test set
# -----------------------------
X_test_tfidf = vectorizer.transform(X_test)
y_pred = clf.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.81      0.81     46658
           1       0.81      0.80      0.80     46658

    accuracy                           0.80     93316
   macro avg       0.80      0.80      0.80     93316
weighted avg       0.80      0.80      0.80     93316


Confusion Matrix:
[[37875  8783]
 [ 9498 37160]]


# Random Forest

In [11]:
# -----------------------------
# 1. Load dataset
# -----------------------------
# Load TSV file
df = pd.read_csv("/kaggle/input/datasets/mrsikandarali/balanced-dataset/balanced_dataset.csv")

X = df['text']
y = df['label']

In [12]:
import time
# -----------------------------
# 4. Initialize Random Forest with warm_start
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=10,  # Start with 10 trees
    max_depth=None,
    max_features='sqrt',
    n_jobs=-1,
    warm_start=True,
    random_state=42
)

In [13]:
# -----------------------------
# 2. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# 3. TF-IDF Vectorization
# -----------------------------
vectorizer = TfidfVectorizer(
    max_features=30000,  # Keep smaller for Random Forest
    ngram_range=(1,2),
    stop_words='english'
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf= vectorizer.transform(X_test)

In [14]:
# -----------------------------
# 5. Train Random Forest incrementally
# -----------------------------
total_trees = 100   # Total trees you want
trees_per_step = 10 # Add 10 trees at a time

print("Starting Random Forest training...")

start_time_total = time.time()

while rf.n_estimators <= total_trees:
    start_time_step = time.time()
    
    rf.fit(X_train_tfidf, y_train)
    
    end_time_step = time.time()
    print(f" Trained {rf.n_estimators} trees (step took {end_time_step - start_time_step:.2f}s)")
    
    rf.n_estimators += trees_per_step  # Add more trees for next step

end_time_total = time.time()
print(f"Random Forest training completed in {end_time_total - start_time_total:.2f} seconds")



Starting Random Forest training...
 Trained 10 trees (step took 193.59s)
 Trained 20 trees (step took 193.38s)
 Trained 30 trees (step took 193.46s)
 Trained 40 trees (step took 194.35s)
 Trained 50 trees (step took 195.94s)
 Trained 60 trees (step took 194.70s)
 Trained 70 trees (step took 196.27s)
 Trained 80 trees (step took 193.56s)
 Trained 90 trees (step took 195.94s)
 Trained 100 trees (step took 193.51s)
Random Forest training completed in 1944.71 seconds


In [15]:
# -----------------------------
# 6. Evaluate
# -----------------------------
y_pred = rf.predict(X_test_tfidf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.81      0.84     46658
           1       0.82      0.87      0.84     46658

    accuracy                           0.84     93316
   macro avg       0.84      0.84      0.84     93316
weighted avg       0.84      0.84      0.84     93316


Confusion Matrix:
[[37978  8680]
 [ 6254 40404]]
